# Wolt menu data

Inspects and cleans the Wolt menus collected with `wolt_menus.py` (Copenhagen bounds, 2 km grid, collected 2026-10-22).

The items are split into separate dataframes, so each group can be inspected on its own:

| dataframe | what |
|---|---|
| `zero_price` | items priced 0 kr (cutlery, info rows, free extras) |
| `caterers` | every item from catering venues |
| `non_food` | bags, cutlery and other packaging that costs money |
| `bundles` | menus, combos, deals and "dish + drink" offers |
| `drinks` | soft drinks, beer, wine, coffee, tea, juice, shakes, ... |
| `food` | what is left: the food-specific menu items |

The order matters: an item only lands in the first group it matches, e.g. "Pasta + sodavand 0,33 l" is a bundle, not a drink.
Drinks and bundles are found with keyword rules on the category and item name, so they are heuristics: the check cells under each step show what was caught and what slipped through.

## Imports

In [4]:
import re
import unicodedata

import pandas as pd
from config import WOLT_DATA_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

## Load

In [5]:
AREA = "vesterbro"

venues = pd.read_csv(WOLT_DATA_DIR / AREA / "venues.csv")
items_raw = pd.read_csv(WOLT_DATA_DIR / AREA / "items.csv")
print(f"{len(venues)} venues, {len(items_raw)} items from {items_raw['slug'].nunique()} venues")
items_raw.head()

193 venues, 10079 items from 193 venues


,slug,venue,category,item,description,price_dkk
0,1001-nat-pizza,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0
1,1001-nat-pizza,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0
2,1001-nat-pizza,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0
3,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0
4,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0


## Helpers

All matching is done on lowercased, unicode-normalised text (`cat_text`, `item_text`).

Matching uses python's `re` instead of `Series.str.contains`: pandas 3 runs `str.contains` on pyarrow's regex engine, where `\b` only knows ASCII letters, so `\bøl\b` never matches "øl".

In [6]:
def normalise(s: pd.Series) -> pd.Series:
    return s.fillna("").map(lambda x: unicodedata.normalize("NFC", x).lower().strip())


def matches(s: pd.Series, pattern: str) -> pd.Series:
    rx = re.compile(pattern)
    return s.map(lambda x: bool(rx.search(x)))


def litres(item_text: pd.Series) -> pd.Series:
    """The volume in the item name in litres ('0,33 l', '33 cl', '330 ml'), NaN if there is none."""
    volume = item_text.str.extract(r"(\d+(?:[.,]\d+)?)\s?(l|cl|ml)\b")
    return volume[0].str.replace(",", ".").astype(float) * volume[1].map({"l": 1, "cl": 0.01, "ml": 0.001})


items = items_raw.assign(cat_text=normalise(items_raw["category"]), item_text=normalise(items_raw["item"]))

## 1. Items priced 0 kr

In [7]:
is_zero = items["price_dkk"] == 0
zero_price = items[is_zero]
items = items[~is_zero]
print(f"{len(zero_price)} items priced 0 kr removed, {len(items)} left")
zero_price["cat_text"].value_counts().head(15)

125 items priced 0 kr removed, 9954 left


cat_text
engangsservice                  104
sådan fungerer wolt catering      6
cutlery                           6
tacos                             2
                                  2
small dishes                      1
fries & dips                      1
build-your-own                    1
engangsbestik og servietter       1
wedos egne dressinger             1
Name: count, dtype: int64

## 2. Caterers

A venue is a caterer if its name or one of its categories mentions "catering". All of its items go to `caterers`.

In [8]:
caterer_slugs = set(items.loc[matches(items["venue"].str.lower(), "catering") | matches(items["cat_text"], "catering"), "slug"])
is_caterer = items["slug"].isin(caterer_slugs)
caterers = items[is_caterer]
items = items[~is_caterer]
print(f"{len(caterer_slugs)} caterers with {len(caterers)} items, {len(items)} items left")
caterers.groupby("venue").agg(items=("item", "size"), median_price=("price_dkk", "median"), max_price=("price_dkk", "max")).sort_values("items", ascending=False)

3 caterers with 83 items, 9871 items left


,items,median_price,max_price
venue,,,
Hallernes Smørrebrød Catering,37,52.0,8000.0
Fromberg Catering,28,35.0,339.0
Frederiksberg Smørrebrød Catering,18,75.0,11550.0


## 3. Non-food (bags, cutlery, packaging)

In [9]:
NON_FOOD = r"engangs|bestik|cutlery|\bposer?\b|bærepose|\bpant\b|emballage|^service$"

is_non_food = matches(items["cat_text"], NON_FOOD)
non_food = items[is_non_food]
items = items[~is_non_food]
print(f"{len(non_food)} non-food items, {len(items)} left")
non_food["cat_text"].value_counts().head(10)

77 non-food items, 9794 left


cat_text
engangsservice                                  75
pose                                             1
engangsservice(huske at bestille spisepinde)     1
Name: count, dtype: int64

## 4. Bundles

Menus, combos, deals and offers, found by category name ("Menuer", "Combos", "Tilbud", "Børnemenu", ...) or item name ("Big Mac Menu", "Ramen + Soft Drink").

In [10]:
BUNDLE_CATEGORY = r"menu|combo|bundle|tilbud|\bdeals?\b|happy meal"
BUNDLE_ITEM = BUNDLE_CATEGORY + r"|\+.*(cola|pepsi|fanta|sprite|sodavand|soft ?drink|drik|øl\b)"

by_category = matches(items["cat_text"], BUNDLE_CATEGORY)
by_item = matches(items["item_text"], BUNDLE_ITEM)
is_bundle = by_category | by_item
bundles = items[is_bundle]
items = items[~is_bundle]
print(f"{len(bundles)} bundles ({by_category.sum()} by category, {(by_item & ~by_category).sum()} only by item name), {len(items)} left")
bundles["cat_text"].value_counts().head(20)

928 bundles (738 by category, 190 only by item name), 8866 left


cat_text
menuer                    254
combos                     82
sushimenuer                36
max classics               32
menus                      28
bundles                    26
børnemenuer                22
green family               22
frokostmenu                20
kylling                    20
mini menu                  18
børnemenu                  15
frokost                    13
menu                       13
wedo combo deals           13
tilbudsmenuer              12
chicken tenders            12
oksekød                    12
happy meal®                12
kylling, fisk & veggie     12
Name: count, dtype: int64

Check: bundles found only by the item name

In [11]:
bundles[~matches(bundles["cat_text"], BUNDLE_CATEGORY)].sample(20, random_state=0)[["venue", "category", "item", "price_dkk"]]

,venue,category,item,price_dkk
6178,MAX Premium Burgers Dybbølsbro,MAX CLASSICS,Cheese 'n' Bacon menu,143.0
6056,MAX Premium Burgers Hovedbanegården,GREEN FAMILY,Crispy Supreme Sandwich menu,139.0
6395,McDonald's Hovedbanegården,Oksekød,Big Tasty® Bacon Menu,128.0
6119,MAX Premium Burgers Hovedbanegården,CHICKEN TENDERS,Chicken Tenders Hot Habanero 3 stk menu,137.0
6198,MAX Premium Burgers Dybbølsbro,KYLLING,Grilled Chicken Buffalo menu,125.0
6047,MAX Premium Burgers Hovedbanegården,MAX CLASSICS,Frisco menu,127.0
2381,El Amigo Vesterbro,Vegetarisk/Vegansk,Burrito menu,150.0
6199,MAX Premium Burgers Dybbølsbro,GREEN FAMILY,Crispy Supreme Buffalo menu,139.0
6292,MAX Premium Burgers Dybbølsbro,KYLLING,Chicken Nuggets 6 stk menu,109.0
960,Börger,Burgere,BÖF Bearnasie Burger Menu,190.0


## 5. Drinks

An item is a drink if
- its category is a drinks category (sodavand, øl, vin, kaffe, juice, smoothies, ...), or
- its name is a drink (latte, espresso, tea, milkshake, ...) or has a drink-sized volume (0.2 - 2 l, e.g. "0,33 l"), unless the name or category says it is food (ice cream sold per litre, dips in ml, curries in "650 ml", ...)

In [12]:
DRINK_CATEGORY = (
    r"sodavand|\bvand\b|\bøl\b|\bbeers?\b|drik|\bdrinks?\b|softdrink|juice|smoothie|shake|lemonade|limonade|\bsaft\b|"
    r"(rød|hvid|rosé|mouserende |hede|\b)vine?\b|\brosé?\b|bobler|champagne|cocktail|spiritus|alkohol|cider|\bsake\b|"
    r"kaffe|coffee|espresso|\blatte|matcha|\bte\b|\btea\b|iste\b|chai|bubble|kakao|kombucha|\blassi\b|ayran|ramune|vitamin well|beverage|\bsoda"
)
DRINK_ITEM = r"\blatte\b|cappuccino|espresso|\bamericano\b|macchiato|\bmocha\b|frap|\bkaffe\b|\bcoffee\b|\btea\b|\bte\b|smoothie|milkshake|lemonade|\bjuice\b"
NOT_DRINK = (
    r"\bis\b|isboks|softice|flødeis|ice ?cream|gelato|ben & jerry|\bdip|dressing|sauce|mayo|bowl|suppe|soup|kage|cake|cookie|"
    r"burger|pizza|sandwich|med ris|raita|curry|vindalo|paneer|masala|oats|\bball\b|\bbar\b"
)

by_category = matches(items["cat_text"], DRINK_CATEGORY)
looks_like_food = matches(items["item_text"], NOT_DRINK) | matches(items["cat_text"], NOT_DRINK)
by_item = (matches(items["item_text"], DRINK_ITEM) | litres(items["item_text"]).between(0.2, 2.0)) & ~looks_like_food
is_drink = by_category | by_item
drinks = items[is_drink]
items = items[~is_drink]
print(f"{len(drinks)} drinks ({by_category.sum()} by category, {(by_item & ~by_category).sum()} only by item name), {len(items)} left")
drinks["cat_text"].value_counts().head(25)

2359 drinks (2269 by category, 90 only by item name), 6507 left


cat_text
sodavand          596
øl                206
vand              150
drinks             94
varme drikke       84
rødvin             71
kolde drikke       69
kaffe & te         69
hvidvin            64
drikkevarer        55
drikkevare         43
kaffe              37
soft drinks        33
juice              31
dåse sodavand      30
milkshakes         28
juicer             24
beer               22
frapino            21
milkshake          20
ready to drink     20
vin                19
lemonade           18
iskaffe            15
signatur-juice     15
Name: count, dtype: int64

Check: drinks found only by the item name

In [13]:
drinks[~matches(drinks["cat_text"], DRINK_CATEGORY)].sample(20, random_state=0)[["venue", "category", "item", "price_dkk"]]

,venue,category,item,price_dkk
1271,Burger Palace,Kiosk,Jack Daniel's Whiskey 70 cl,349.0
2464,Espresso House Fisketorvet,Frapino,Coffee Caramel Frapino Std,74.0
8324,Sober Boba Bubble Tea,Cheese Foam 0.7L,Cheese Foam Matcha Milk Tea,101.0
8183,Smagsløget Vesterbro,Cocio,"Cocio 0,4 l",30.0
8405,Starbucks® Fisketorvet,Seasonal Favorites,Iced Caramel Protein Latte,62.0
2615,Espresso House Tivoli,Frapino,Coffee Caramel Frapino Std,74.0
8302,Sober Boba Bubble Tea,Brown Sugar,Brown Sugar Pearl Black Tea Latte,82.0
8300,Sober Boba Bubble Tea,Brown Sugar,Brown Sugar Milk Tea,69.7
8450,Starbucks® Fisketorvet,Mocha,Iced Caffé Mocha,62.0
9589,Warung,Noda,"Noda Mango Passionsfrugt 0,33 l",39.0


## 6. Food

In [14]:
items.head()

,slug,venue,category,item,description,price_dkk,cat_text,item_text
0,1001-nat-pizza,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0,burger,klassisk okseburger
1,1001-nat-pizza,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0,hovedretter,en halv grillkylling
2,1001-nat-pizza,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0,burger,maxi cheeseburger
3,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0,pizzaer,gorgonzola
4,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0,pizzaer,caprese


In [15]:
food = items.drop(columns=["cat_text", "item_text"])
print(f"{len(food)} food items from {food['slug'].nunique()} venues")
food.head()

6507 food items from 189 venues


,slug,venue,category,item,description,price_dkk
0,1001-nat-pizza,1001 Nat Pizza,Burger,Klassisk okseburger,"Sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0
1,1001-nat-pizza,1001 Nat Pizza,Hovedretter,En halv grillkylling,NaN,108.0
2,1001-nat-pizza,1001 Nat Pizza,Burger,Maxi Cheeseburger,"Briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0
3,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Gorgonzola,"Tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0
4,1001-nat-pizza,1001 Nat Pizza,Pizzaer,Caprese,"Tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0


Check: food items that still look like drinks (what the rules missed)

In [16]:
LEFTOVER = r"\bcola\b|coca|pepsi|fanta|\bøl\b|\bvin\b|juice|kaffe|latte|smoothie|\btea\b|\bte\b|0,33|\bcl\b"
leftovers = items[matches(items["item_text"], LEFTOVER)]
print(f"{len(leftovers)} food items match drink words ({len(leftovers) / len(food):.2%} of food)")
leftovers.sample(min(20, len(leftovers)), random_state=0)[["venue", "category", "item", "price_dkk"]]

4 food items match drink words (0.06% of food)


,venue,category,item,price_dkk
7741,Restaurant Aquavit,Smørrebrød,Platte,149.0
8974,The South Indian Vesterbro,Foretter,13. Kings Platter 65,149.0
6979,Namaste Fisketorvet,Forretter,Tandoori Platter,149.0
3518,Great China Istedgade,Dry-wok retter,一百零七 Tørstegt and med øl,188.0


In [17]:
food["category"].value_counts().head(30)

category
Tilbehør           405
Hovedretter        327
Pizzaer            324
Forretter          290
Dyppelse           207
Dips               139
Sides              130
Dim Sum            127
Pizze              120
Pizza              111
Nigiri             111
Sandwiches         102
Uramaki            102
Salater             91
Kaburimaki          79
Desserter           73
Burgere             61
Hosomaki            61
Smørrebrød          60
Futomaki            56
Pizza               52
Sticks              52
Pasta               50
TILLBEHØR & DIP     50
Burger              44
Snacks              44
Kylling             40
Kyllingeretter      38
Bagværk & Kage      36
Lammeretter         36
Name: count, dtype: int64

In [18]:
food["price_dkk"].describe()

count    6507.000000
mean       96.864069
std        55.853340
min         1.000000
25%        50.000000
50%        96.000000
75%       135.000000
max       659.000000
Name: price_dkk, dtype: float64

## Summary

In [19]:
splits = {"zero_price": zero_price, "caterers": caterers, "non_food": non_food, "bundles": bundles, "drinks": drinks, "food": food}
summary = pd.DataFrame({
    "items": {name: len(df) for name, df in splits.items()},
    "venues": {name: df["slug"].nunique() for name, df in splits.items()},
    "median_price_dkk": {name: df["price_dkk"].median() for name, df in splits.items()},
})
summary["share_of_items"] = (summary["items"] / len(items_raw)).round(3)
assert summary["items"].sum() == len(items_raw), "every item should be in exactly one dataframe"
summary

,items,venues,median_price_dkk,share_of_items
zero_price,125,77,0.0,0.012
caterers,83,3,52.0,0.008
non_food,77,47,4.0,0.008
bundles,928,113,166.0,0.092
drinks,2359,179,40.0,0.234
food,6507,189,96.0,0.646
